In [2]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import duckdb

atl = pd.read_csv("atl_flights_2023_clean.csv")
print(atl.shape)

(665876, 27)


In [15]:
monthly = duckdb.sql("""
    SELECT Month,
           Month_Name,
           COUNT(*) AS flights,
           ROUND(AVG(Dep_Delay_15) * 100, 1) AS pct_dep_delayed,
           MEDIAN(Dep_Delay) AS median_dep_delay
    FROM atl
    GROUP BY Month, Month_Name
    ORDER BY Month
""").df()

In [16]:
causes = duckdb.sql("""
    WITH cause_totals AS (
        SELECT 'Carrier' AS cause, SUM(Delay_Carrier) AS minutes FROM atl
        UNION ALL
        SELECT 'Late Aircraft', SUM(Delay_LastAircraft) FROM atl
        UNION ALL
        SELECT 'NAS (Air Traffic System)', SUM(Delay_NAS) FROM atl
        UNION ALL
        SELECT 'Weather', SUM(Delay_Weather) FROM atl
        UNION ALL
        SELECT 'Security', SUM(Delay_Security) FROM atl
    )
    SELECT cause,
           minutes,
           ROUND(minutes * 100.0 / SUM(minutes) OVER (), 1) AS pct_of_delay_minutes
    FROM cause_totals
    ORDER BY minutes DESC
""").df()

causes

,cause,minutes,pct_of_delay_minutes
0,Carrier,3674355.0,44.7
1,Late Aircraft,2657708.0,32.4
2,NAS (Air Traffic System),1394128.0,17.0
3,Weather,472589.0,5.8
4,Security,13966.0,0.2


In [6]:
causes_by_month = duckdb.sql("""
    WITH monthly AS (
        SELECT Month,
               Month_Name,
               SUM(Delay_Carrier)      AS carrier,
               SUM(Delay_LastAircraft) AS late_aircraft,
               SUM(Delay_NAS)          AS nas,
               SUM(Delay_Weather)      AS weather,
               SUM(Delay_Security)     AS security
        FROM atl
        GROUP BY Month, Month_Name
    )
    SELECT Month,
           Month_Name,
           carrier + late_aircraft + nas + weather + security AS total_minutes,
           ROUND(carrier       * 100.0 / (carrier + late_aircraft + nas + weather + security), 1) AS pct_carrier,
           ROUND(late_aircraft * 100.0 / (carrier + late_aircraft + nas + weather + security), 1) AS pct_late_aircraft,
           ROUND(nas           * 100.0 / (carrier + late_aircraft + nas + weather + security), 1) AS pct_nas,
           ROUND(weather       * 100.0 / (carrier + late_aircraft + nas + weather + security), 1) AS pct_weather
    FROM monthly
    ORDER BY Month
""").df()

causes_by_month

,Month,Month_Name,total_minutes,pct_carrier,pct_late_aircraft,pct_nas,pct_weather
0,1,January,800305.0,41.8,30.0,18.0,10.0
1,2,February,418776.0,52.6,28.0,13.6,5.7
2,3,March,818710.0,43.8,31.0,18.9,6.1
3,4,April,746363.0,45.2,32.4,17.6,4.5
4,5,May,560530.0,51.1,32.2,13.4,3.1
5,6,June,1090744.0,43.2,34.4,16.9,5.5
6,7,July,1135888.0,42.8,35.0,16.7,5.4
7,8,August,1022132.0,37.3,33.9,20.3,8.4
8,9,September,504180.0,46.4,29.1,18.8,5.6
9,10,October,386300.0,51.6,32.9,12.2,3.1


In [7]:
routes = duckdb.sql("""
    SELECT Arr_Airport,
           Arr_CityName,
           COUNT(*) AS flights,
           ROUND(AVG(Dep_Delay_15) * 100, 1) AS pct_dep_delayed
    FROM atl
    WHERE Dep_Airport = 'ATL'
    GROUP BY Arr_Airport, Arr_CityName
    HAVING COUNT(*) >= 1000
    ORDER BY pct_dep_delayed DESC
    LIMIT 10
""").df()

routes

,Arr_Airport,Arr_CityName,flights,pct_dep_delayed
0,JFK,"New York, NY",3736,31.7
1,FLL,"Fort Lauderdale, FL",7900,29.8
2,LAS,"Las Vegas, NV",5080,28.4
3,MIA,"Miami, FL",6711,27.9
4,SJU,"San Juan, PR",1901,27.6
5,MCO,"Orlando, FL",8960,27.2
6,BWI,"Baltimore, MD",6379,26.5
7,PHL,"Philadelphia, PA",5767,26.1
8,SFO,"San Francisco, CA",3241,25.9
9,TPA,"Tampa, FL",6615,25.8


In [8]:
airlines = duckdb.sql("""
    SELECT Airline,
           COUNT(*) AS flights,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_atl_departures,
           ROUND(AVG(Dep_Delay_15) * 100, 1) AS pct_dep_delayed,
           MEDIAN(Dep_Delay) AS median_dep_delay
    FROM atl
    WHERE Dep_Airport = 'ATL'
    GROUP BY Airline
    HAVING COUNT(*) >= 1000
    ORDER BY flights DESC
""").df()

airlines

,Airline,flights,pct_of_atl_departures,pct_dep_delayed,median_dep_delay
0,Delta Air Lines Inc,217802,65.5,19.0,-1.0
1,Southwest Airlines Co.,36415,11.0,25.6,2.0
2,Endeavor Air,20434,6.1,11.5,-4.0
3,Spirit Air Lines,13440,4.0,31.2,2.0
4,Frontier Airlines Inc.,10226,3.1,42.3,8.0
5,Skywest Airlines Inc.,9672,2.9,17.9,-2.0
6,American Airlines Inc.,8485,2.6,19.1,-4.0
7,United Air Lines Inc.,7948,2.4,22.3,-1.0
8,JetBlue Airways,3497,1.1,34.5,2.0
9,Republic Airways,2162,0.7,10.8,-5.0


In [9]:
time_of_day = duckdb.sql("""
    SELECT DepTime_label,
           COUNT(*) AS flights,
           ROUND(AVG(Dep_Delay_15) * 100, 1) AS pct_dep_delayed,
           ROUND(SUM(Delay_LastAircraft) * 100.0 /
                 SUM(Delay_Carrier + Delay_LastAircraft + Delay_NAS + Delay_Weather + Delay_Security), 1)
                 AS pct_late_aircraft_minutes
    FROM atl
    WHERE Dep_Airport = 'ATL'
    GROUP BY DepTime_label
    ORDER BY CASE DepTime_label
                 WHEN 'Morning'   THEN 1
                 WHEN 'Afternoon' THEN 2
                 WHEN 'Evening'   THEN 3
                 WHEN 'Night'     THEN 4
             END
""").df()

time_of_day

,DepTime_label,flights,pct_dep_delayed,pct_late_aircraft_minutes
0,Morning,117917,14.0,19.8
1,Afternoon,109140,21.1,26.7
2,Evening,104052,27.7,34.5
3,Night,1826,15.2,17.4


In [17]:
monthly.to_csv("results_monthly.csv", index=False)
causes.to_csv("results_causes.csv", index=False)
causes_by_month.to_csv("results_causes_by_month.csv", index=False)
routes.to_csv("results_routes.csv", index=False)
airlines.to_csv("results_airlines.csv", index=False)
time_of_day.to_csv("results_time_of_day.csv", index=False)